# Lab 1, Notebook 2: Credentials and Cypher Basics

This notebook does the setup every later lab depends on. You enter your Neo4j Aura credentials once, here, and store them in a Databricks secret scope. Labs 2 through 6 read that scope, so this is the only notebook in the whole workshop where you type a Neo4j password.

Then you build a tiny aircraft graph by hand, query it, and delete it. That is enough Cypher to read the queries in Labs 2 through 6.

## What You'll Learn
- How to connect to Aura from a Databricks notebook with the Neo4j Python driver
- How to store credentials in a Databricks secret scope the rest of the workshop reads
- How to create nodes and relationships with Cypher
- How to read the graph back with MATCH, RETURN, and WHERE
- How to clear the graph before Lab 2

## Prerequisites
- `01_create_aura_instance.ipynb` finished, with the credentials file downloaded
- Access to the workshop Databricks cluster (the `neo4j` Python driver is already installed on it)

## Instructions
1. Clone this notebook to your personal folder
2. Enter your Neo4j credentials in the Configuration cell
3. Run the remaining cells in order (Shift+Enter)


## Configuration

Enter the values from the credentials file you downloaded in notebook 01.

`NEO4J_URI` is the connection string that starts with `neo4j+s://`. `NEO4J_USERNAME` is `neo4j` unless you changed it. `NEO4J_PASSWORD` is the generated password from the downloaded file. `NEO4J_DATABASE` is the `NEO4J_DATABASE` line in that same file, usually `neo4j`.

> **Attach to the classic compute, not serverless.** A classic all-purpose cluster was created for your workspace, and it is the only compute with the workshop libraries (the `neo4j` Python driver and the Neo4j Spark Connector) installed. Serverless does not have them, so the notebook fails there.
>
> Pick it from the compute selector at the top right of the notebook. To see its name and state, open **Compute** in the left menu, then the **All-purpose compute** tab. If it is stopped, starting it takes a couple of minutes; wait for the state to reach **Running** before you run any cell.


In [ ]:
# ============================================
# CONFIGURATION - Enter your Neo4j credentials
# ============================================

NEO4J_URI = ""  # e.g., "neo4j+s://xxxxxxxx.databases.neo4j.io"
NEO4J_USERNAME = "neo4j"
NEO4J_PASSWORD = ""  # Your password from the downloaded credentials file
NEO4J_DATABASE = "neo4j"  # NEO4J_DATABASE from the downloaded credentials file

# Validate configuration
if not NEO4J_URI or not NEO4J_PASSWORD or not NEO4J_DATABASE:
    print("WARNING: Please enter your Neo4j credentials above before running the notebook!")
else:
    print("Configuration ready!")
    print(f"Neo4j URI: {NEO4J_URI}")
    print(f"Neo4j database: {NEO4J_DATABASE}")


### Test the Connection

The cell below opens a driver, calls `verify_connectivity()`, and runs one trivial query against `NEO4J_DATABASE`. A server version printed back means the URI, username, password, and database name are all correct.

The `neo4j` Python driver is already installed as a cluster library, so there is no `pip install` step.

Common failures:
- **`AuthError`**: the password is wrong. Re-read it from the downloaded file.
- **`ServiceUnavailable`**: the URI is wrong, or the instance is still starting. Check the console shows "RUNNING".
- **`Neo.ClientError.Database.DatabaseNotFound`**: the database name is wrong. Copy `NEO4J_DATABASE` from the downloaded credentials file.
- **`ValueError` from this cell**: the Configuration cell above is still blank.


In [ ]:
from neo4j import GraphDatabase

if not NEO4J_URI.strip() or not NEO4J_PASSWORD.strip() or not NEO4J_DATABASE.strip():
    raise ValueError(
        "NEO4J_URI, NEO4J_PASSWORD and NEO4J_DATABASE are not all set. Enter your "
        "Aura credentials in the Configuration cell above, then run this cell again."
    )

with GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD)) as test_driver:
    test_driver.verify_connectivity()
    _, summary, _ = test_driver.execute_query("RETURN 1 AS ok", database_=NEO4J_DATABASE)

print("Connected to Neo4j Aura")
print(f"Server:   {summary.server.address}")
print(f"Version:  {summary.server.agent}")
print(f"Database: {NEO4J_DATABASE}")


## Store the Credentials in a Secret Scope

The Configuration cell above is the only place in this workshop where you type your Neo4j password, and this cell is what makes that true. It writes the four values into a Databricks secret scope. Labs 2 through 6 read them from that scope instead of asking again.

Scope names are unique per workspace rather than per user, so one literal name would collide across a shared workshop workspace. The cell derives `fleet-ops-<your user>` from `current_user()` and prints the result.

The scope holds four keys:

| Key | Value |
|-----|-------|
| `neo4j-uri` | Your Aura connection URI, `neo4j+s://xxxxxxxx.databases.neo4j.io` |
| `neo4j-username` | Usually `neo4j` |
| `neo4j-password` | Your password from notebook 01 |
| `neo4j-database` | The database name from your credentials file, usually `neo4j` |

The cell is safe to re-run. It reuses an existing scope and overwrites the four keys. It refuses to run at all if the URI, the password, or the database name is missing, so a re-run after clearing the Configuration cell cannot replace working credentials with placeholders.


In [ ]:
import re

from databricks.sdk import WorkspaceClient
from databricks.sdk.errors import ResourceAlreadyExists

# These names are mirrored in Lab_3_Semantic_Search/data_utils.py and in
# lab/workshop.py. Labs 2 through 6 read this scope with these key names, so all
# three places have to agree exactly.
SECRET_SCOPE_PREFIX = "fleet-ops"
SECRET_KEY_NEO4J_URI = "neo4j-uri"
SECRET_KEY_NEO4J_USERNAME = "neo4j-username"
SECRET_KEY_NEO4J_PASSWORD = "neo4j-password"
SECRET_KEY_NEO4J_DATABASE = "neo4j-database"

# Databricks caps a scope name at 128 characters. The prefix plus its separator
# take 10, leaving this much for the user slug.
MAX_SCOPE_SLUG_LENGTH = 118

user = spark.sql("SELECT current_user()").collect()[0][0]
slug = re.sub(r"[^a-z0-9]+", "-", user.lower()).strip("-")[:MAX_SCOPE_SLUG_LENGTH]
SECRET_SCOPE = f"{SECRET_SCOPE_PREFIX}-{slug}"
print(f"Secret scope: {SECRET_SCOPE}")

# Refuse to overwrite a working secret with a placeholder. Without this guard, a
# re-run after clearing the Configuration cell would wipe the stored credentials.
PLACEHOLDER_URI = "neo4j+s://xxxxxxxx.databases.neo4j.io"
if not NEO4J_URI.strip() or NEO4J_URI.strip() == PLACEHOLDER_URI:
    raise ValueError(
        "NEO4J_URI is not set. Enter your Aura connection URI in the "
        "Configuration cell above, then run this cell again."
    )
if not NEO4J_PASSWORD.strip():
    raise ValueError(
        "NEO4J_PASSWORD is not set. Enter your Aura password in the "
        "Configuration cell above, then run this cell again."
    )
if not NEO4J_DATABASE.strip():
    raise ValueError(
        "NEO4J_DATABASE is not set. Enter your Aura database name in the "
        "Configuration cell above, then run this cell again."
    )

w = WorkspaceClient()
try:
    w.secrets.create_scope(scope=SECRET_SCOPE)
    print(f"Created scope: {SECRET_SCOPE}")
except ResourceAlreadyExists:
    print(f"Scope already exists, reusing it: {SECRET_SCOPE}")

w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_URI, string_value=NEO4J_URI.strip())
w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_USERNAME, string_value=NEO4J_USERNAME.strip())
w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_PASSWORD, string_value=NEO4J_PASSWORD)
w.secrets.put_secret(scope=SECRET_SCOPE, key=SECRET_KEY_NEO4J_DATABASE, string_value=NEO4J_DATABASE.strip())

print(
    f"Stored keys: {SECRET_KEY_NEO4J_URI}, {SECRET_KEY_NEO4J_USERNAME}, "
    f"{SECRET_KEY_NEO4J_PASSWORD}, {SECRET_KEY_NEO4J_DATABASE}"
)


### Read the Credentials Back

Reading from the scope now proves it is readable before six notebooks depend on it. This is the same read every later notebook does, and the rest of this notebook uses the values it returns.

Databricks redacts secret values in notebook output, so the URI below prints as `[REDACTED]`. That is expected, not a bug. The value is intact in Python and the Neo4j connection works.


In [ ]:
NEO4J_URI = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY_NEO4J_URI)
NEO4J_USERNAME = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY_NEO4J_USERNAME)
NEO4J_PASSWORD = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY_NEO4J_PASSWORD)
NEO4J_DATABASE = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY_NEO4J_DATABASE)

print(f"Read credentials from scope: {SECRET_SCOPE}")
print(f"Neo4j URI: {NEO4J_URI}")
print(f"Neo4j database: {NEO4J_DATABASE}")


## Cypher Basics

Cypher is Neo4j's query language. It draws patterns. `(a:Aircraft)` is a node, `-[:HAS_SYSTEM]->` is a relationship, and a query is those pieces joined into a shape you want to find or create.

Before you load real data in Lab 2, build a two-node graph by hand and query it.

### Two Ways to Run These Queries

Every query below runs from this notebook through the Python driver. The same text also runs unchanged in the Aura Query editor, which shows results as a picture of the graph rather than a table:

1. Go to [console.neo4j.io](https://console.neo4j.io)
2. Select your instance
3. Click **Query** to open the query editor

Run each query here first, then paste it into the Query editor to see the same result drawn as nodes and lines.


### A Helper for Running Cypher

One driver and one function, defined once and reused by every cell below. `run_cypher` sends a Cypher string to Aura, reports what changed, and returns the rows as a pandas DataFrame.

In [ ]:
import pandas as pd
from neo4j import GraphDatabase
from neo4j.graph import Node, Relationship

# One driver for the rest of the notebook. Closed in the cleanup section.
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USERNAME, NEO4J_PASSWORD))
driver.verify_connectivity()


def _to_cell(value):
    """Flatten graph objects into something a DataFrame column can hold."""
    if isinstance(value, Node):
        return dict(value)
    if isinstance(value, Relationship):
        return f"[:{value.type}]"
    return value


def run_cypher(query, **params):
    """Run a Cypher query and return the rows as a pandas DataFrame."""
    records, summary, keys = driver.execute_query(query, database_=NEO4J_DATABASE, **params)

    c = summary.counters
    changes = {
        "nodes created": c.nodes_created,
        "relationships created": c.relationships_created,
        "properties set": c.properties_set,
        "nodes deleted": c.nodes_deleted,
        "relationships deleted": c.relationships_deleted,
    }
    changed = ", ".join(f"{name}: {n}" for name, n in changes.items() if n)
    print(changed if changed else "no changes to the graph")

    if not records:
        print("no rows returned")
        return None
    return pd.DataFrame([[_to_cell(r[k]) for k in keys] for r in records], columns=keys)


print("Helper ready.")

### Creating Nodes

Create an Aircraft node with properties:

`CREATE` makes a node. `:Aircraft` is a **label** (like a type). Properties go inside curly braces.

In [ ]:
run_cypher("""
    CREATE (a:Aircraft {tail_number: 'N12345', model: 'B737-800', manufacturer: 'Boeing'})
    RETURN a
""")

### Reading Nodes

Find all Aircraft nodes and return their properties:

`MATCH` finds patterns in the graph. `RETURN` selects what to display.

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)
    RETURN a.tail_number, a.model, a.manufacturer
""")

### Creating Relationships

Create two nodes connected by a relationship:

`-[:HAS_SYSTEM]->` creates a directed relationship from the Aircraft to the System.

This adds a second Aircraft node with the same tail number. That is fine here. Lab 2 adds uniqueness constraints so the real dataset cannot pick up duplicates.

In [ ]:
run_cypher("""
    CREATE (a:Aircraft {tail_number: 'N12345', model: 'B737-800'})
    CREATE (s:System {name: 'Engine #1', type: 'CFM56-7B'})
    CREATE (a)-[:HAS_SYSTEM]->(s)
    RETURN a, s
""")

### Querying Relationships

Traverse relationships to find connected nodes:

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)-[:HAS_SYSTEM]->(s:System)
    RETURN a.tail_number, s.name, s.type
""")

### Filtering with WHERE

Add conditions to narrow results:

In [ ]:
run_cypher("""
    MATCH (a:Aircraft)
    WHERE a.manufacturer = 'Boeing'
    RETURN a.tail_number, a.model
""")

## Cleanup Before Lab 2

Remove all nodes and relationships to start fresh:

`DETACH DELETE` removes nodes and all their relationships. Plain `DELETE` refuses to remove a node that still has relationships attached, so `DETACH` is what makes this one line enough.

> **Warning:** Run this cleanup before starting Lab 2. Lab 2 expects an empty graph and loads the full Aircraft Digital Twin dataset from scratch. Practice nodes left behind show up in its verification counts.

> **Tip:** These examples are for learning. In Lab 2 you load the full dataset programmatically using the Neo4j Spark Connector.

In [ ]:
run_cypher("MATCH (n) DETACH DELETE n")

# Confirm the graph is empty
print(run_cypher("MATCH (n) RETURN count(n) AS remaining"))

# Done with the driver. Re-run the helper cell above to reopen it.
driver.close()

## Next Steps

Your Aura instance is running and empty, its credentials are in the `fleet-ops-<your user>` secret scope, and you have watched a connection to it succeed.

Continue to **Lab 2 - Databricks ETL to Neo4j** to load the Aircraft Digital Twin dataset into this instance with the Neo4j Spark Connector.

You will not type the password again. Every notebook from Lab 2 on opens with a cell that reads this scope. If one of them stops with an error about a missing scope, come back and run this notebook.
